In [1]:
#Mount Drive and verify GPU
from google.colab import drive
drive.mount('/content/drive')

import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")

# Verify input data exists
import os
import numpy as np

# UPDATED PATH to 70/10/20 folder
stgcn_dir = '/content/drive/MyDrive/HRC_Research/datasets/HRI30/HRI30_70_10_20/stgcn_format'
for fname in ['train_data.npy', 'test_data.npy', 'train_label.pkl', 'test_label.pkl']:
    fpath = os.path.join(stgcn_dir, fname)
    if os.path.exists(fpath):
        size_mb = os.path.getsize(fpath) / 1024**2
        print(f"{fname}: {size_mb:.1f} MB")
    else:
        print(f"MISSING: {fname}")

Mounted at /content/drive
CUDA available: True
GPU: Tesla T4
train_data.npy: 88.3 MB
test_data.npy: 25.2 MB
train_label.pkl: 0.1 MB
test_label.pkl: 0.0 MB


In [2]:
# Install dependencies and clone ST-GCN repo
# Install scikit-video (required by ST-GCN repo)
!pip install scikit-video -q

# Clone ST-GCN repo
import os
if not os.path.exists('/content/st-gcn'):
    !git clone https://github.com/yysijie/st-gcn.git /content/st-gcn
else:
    print("Repo already exists.")

# Install torchlight (ST-GCN's internal package)
%cd /content/st-gcn
!pip install -e torchlight -q

print("Dependencies ready.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 34.4 MB/s eta 0:00:00
Cloning into '/content/st-gcn'...
remote: Enumerating objects: 1523, done.
remote: Total 1523 (delta 0), reused 0 (delta 0), pack-reused 1523 (from 1)
Receiving objects: 100% (1523/1523), 57.17 MiB | 34.17 MiB/s, done.
Resolving deltas: 100% (780/780), done.
/content/st-gcn
  Preparing metadata (setup.py) ... done
Dependencies ready.


In [3]:
# Apply required patches to ST-GCN repo
from pathlib import Path

# torch.load in torchlight/torchlight/io.py
io_path = Path('/content/st-gcn/torchlight/torchlight/io.py')
text = io_path.read_text()
if 'weights_only=False' not in text:
    text = text.replace(
        'torch.load(weights_path)',
        'torch.load(weights_path, weights_only=False, map_location="cpu")'
    )
    io_path.write_text(text)
    print("Patch 1 applied: torch.load")
else:
    print("Patch 1 already applied.")

# yaml.load in processor/recognition.py
proc_path = Path('/content/st-gcn/processor/recognition.py')
text2 = proc_path.read_text()
if 'yaml.safe_load' not in text2:
    text2 = text2.replace('yaml.load(f)', 'yaml.safe_load(f)')
    proc_path.write_text(text2)
    print("Patch 2 applied: yaml.safe_load")
else:
    print("Patch 2 already applied.")

print("All patches done.")

Patch 1 applied: torch.load
Patch 2 applied: yaml.safe_load
All patches done.


In [4]:
# Download pretrained weights
import os

weights_dir = '/content/st-gcn/models'
weights_path = f'{weights_dir}/st_gcn.ntu-xsub.pt'

os.makedirs(weights_dir, exist_ok=True)

if not os.path.exists(weights_path) or os.path.getsize(weights_path) < 1024*1024:
    print("Downloading pretrained weights...")
    !pip install gdown -q
    import gdown
    gdown.download(
        'https://drive.google.com/uc?id=18pcNj4Bu4Ub7S3YJSsRNJ45XxA4GyaYG',
        weights_path,
        quiet=False
    )
else:
    print("Weights already present.")

size_mb = os.path.getsize(weights_path) / 1024**2
print(f"st_gcn.ntu-xsub.pt: {size_mb:.1f} MB")

Downloading...
From: https://drive.google.com/uc?id=18pcNj4Bu4Ub7S3YJSsRNJ45XxA4GyaYG
To: /content/st-gcn/models/st_gcn.ntu-xsub.pt
100%|██████████| 12.5M/12.5M [00:00<00:00, 81.0MB/s]

st_gcn.ntu-xsub.pt: 11.9 MB


In [5]:
# Load data and create DataLoaders
import numpy as np
import pickle
import torch
from torch.utils.data import Dataset, DataLoader

# UPDATED PATH to 70/10/20 folder
stgcn_dir = '/content/drive/MyDrive/HRC_Research/datasets/HRI30/HRI30_70_10_20/stgcn_format'

# Load arrays
X_train = np.load(os.path.join(stgcn_dir, 'train_data.npy'))
X_test  = np.load(os.path.join(stgcn_dir, 'test_data.npy'))

with open(os.path.join(stgcn_dir, 'train_label.pkl'), 'rb') as f:
    _, y_train = pickle.load(f)

with open(os.path.join(stgcn_dir, 'test_label.pkl'), 'rb') as f:
    _, y_test = pickle.load(f)

y_train = np.array(y_train)
y_test  = np.array(y_test)

print("X_train:", X_train.shape, X_train.dtype)
print("X_test:", X_test.shape, X_test.dtype)
print("y_train:", y_train.shape, "classes:", np.unique(y_train))
print("y_test:", y_test.shape)

# Dataset class
class HRI30Dataset(Dataset):
    def __init__(self, data, labels):

        self.data = torch.tensor(data, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.data[idx], self.labels[idx]

train_dataset = HRI30Dataset(X_train, y_train)
test_dataset  = HRI30Dataset(X_test,  y_test)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,  num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

print(f"\nTrain batches: {len(train_loader)}")
print(f"Test batches:  {len(test_loader)}")

# Verify one batch
batch_x, batch_y = next(iter(train_loader))
print(f"Batch shape: {batch_x.shape}")
print(f"Label shape: {batch_y.shape}")

X_train: (2058, 3, 150, 25, 1) float32
X_test: (588, 3, 150, 25, 1) float32
y_train: (2058,) classes: [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
 24 25 26 27 28 29]
y_test: (588,)

Train batches: 65
Test batches:  19
Batch shape: torch.Size([32, 3, 150, 25, 1])
Label shape: torch.Size([32])


In [6]:
# Load ST-GCN model and replace final layer
import sys
import torch.nn as nn
import math

sys.path.insert(0, '/content/st-gcn')

from net.st_gcn import Model as STGCN

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using device:", device)

# Build the model with NTU settings (60 classes) first
model = STGCN(
    in_channels=3,
    num_class=60,
    dropout=0.5,
    edge_importance_weighting=True,
    graph_args={
        'layout': 'ntu-rgb+d',
        'strategy': 'spatial'
    }
)

# Load pretrained NTU weights
state_dict = torch.load(
    '/content/st-gcn/models/st_gcn.ntu-xsub.pt',
    map_location='cpu',
    weights_only=False
)
model.load_state_dict(state_dict)
print("Pretrained NTU weights loaded.")

model.fcn = nn.Conv2d(256, 30, kernel_size=1)
nn.init.normal_(model.fcn.weight, 0, math.sqrt(2. / 30))
print("Final layer replaced: 60 to 30 classes")
print("New fcn layer:", model.fcn)

model = model.to(device)
print("Model on", device)

Using device: cuda
Pretrained NTU weights loaded.
Final layer replaced: 60 to 30 classes
New fcn layer: Conv2d(256, 30, kernel_size=(1, 1), stride=(1, 1))
Model on cuda


In [7]:
# Define optimizer, scheduler, and loss
import torch.optim as optim

backbone_params = [p for name, p in model.named_parameters() if 'fcn' not in name]
head_params     = [p for name, p in model.named_parameters() if 'fcn' in name]

optimizer = optim.SGD([
    {'params': backbone_params, 'lr': 1e-3},
    {'params': head_params,     'lr': 1e-2}
], momentum=0.9, nesterov=True, weight_decay=1e-4)

scheduler = optim.lr_scheduler.MultiStepLR(optimizer, milestones=[30, 40], gamma=0.1)

criterion = nn.CrossEntropyLoss()

print("Optimizer: SGD with momentum")
print("Backbone LR: 1e-3, Head LR: 1e-2")
print("LR decay at epochs 30 and 40")
print("Loss: CrossEntropyLoss")

Optimizer: SGD with momentum
Backbone LR: 1e-3, Head LR: 1e-2
LR decay at epochs 30 and 40
Loss: CrossEntropyLoss


In [8]:
# DIAGNOSTIC: MOCK FORWARD PASS
print("============ DIAGNOSTIC START ============")
model.eval()
try:
    batch_x, batch_y = next(iter(train_loader))
    print(f"Loaded Batch Shape: {batch_x.shape}")

    if len(batch_x.shape) != 5:
        raise ValueError(f"CRITICAL: ST-GCN expects 5 dimensions (B, C, T, V, M). Got {len(batch_x.shape)}")

    batch_x = batch_x.to(device)
    output = model(batch_x)
    print(f"Model Output Shape: {output.shape}")

    if output.shape[1] != 30:
         raise ValueError(f"CRITICAL: Expected 30 HRI30 classes, but model output {output.shape[1]}")

    print("Forward pass successful! No dimensional crashes.")
    print("STATUS: 100% SAFE TO PROCEED TO CELL 8 (TRAINING)!")

except Exception as e:
    print(f"\n CRITICAL ERROR DETECTED:")
    print(f"Type: {type(e).__name__}")
    print(f"Message: {e}")
    print("\nDO NOT proceed to Cell 8. Share this output with me.")

============ DIAGNOSTIC START ============
Loaded Batch Shape: torch.Size([32, 3, 150, 25, 1])
Model Output Shape: torch.Size([32, 30])
Forward pass successful! No dimensional crashes.
STATUS: 100% SAFE TO PROCEED TO CELL 8 (TRAINING)!


In [9]:
# Training loop with checkpoint saving
import time

NUM_EPOCHS = 50
# UPDATED PATH to save new weights safely
CHECKPOINT_DIR = '/content/drive/MyDrive/HRC_Research/checkpoints/stgcn_hri30_70_10_20'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

best_acc = 0.0
best_epoch = 0

for epoch in range(1, NUM_EPOCHS + 1):
    # TRAIN
    model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for batch_x, batch_y in train_loader:
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)

        optimizer.zero_grad()
        output = model(batch_x)
        loss = criterion(output, batch_y)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * batch_x.size(0)
        _, predicted = output.max(1)
        train_correct += predicted.eq(batch_y).sum().item()
        train_total += batch_x.size(0)

    scheduler.step()

    train_loss /= train_total
    train_acc = 100.0 * train_correct / train_total

    # EVAL
    model.eval()
    test_loss = 0.0
    test_correct = 0
    test_total = 0

    with torch.no_grad():
        for batch_x, batch_y in test_loader:
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)
            output = model(batch_x)
            loss = criterion(output, batch_y)
            test_loss += loss.item() * batch_x.size(0)
            _, predicted = output.max(1)
            test_correct += predicted.eq(batch_y).sum().item()
            test_total += batch_x.size(0)

    test_loss /= test_total
    test_acc = 100.0 * test_correct / test_total

    print(f"Epoch {epoch:02d}/{NUM_EPOCHS} | "
          f"Train Loss: {train_loss:.4f} Acc: {train_acc:.2f}% | "
          f"Test Loss: {test_loss:.4f} Acc: {test_acc:.2f}%")

    # CHECKPOINT every 5 epochs
    if epoch % 5 == 0:
        ckpt_path = os.path.join(CHECKPOINT_DIR, f'epoch_{epoch:02d}.pt')
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'train_acc': train_acc,
            'test_acc': test_acc,
        }, ckpt_path)
        print(f"  → Checkpoint saved: epoch_{epoch:02d}.pt")

    # SAVE BEST MODEL
    if test_acc > best_acc:
        best_acc = test_acc
        best_epoch = epoch
        best_path = os.path.join(CHECKPOINT_DIR, 'best_stgcn_hri30.pt')
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'test_acc': test_acc,
        }, best_path)
        print(f"  → New best model saved! Test Acc: {best_acc:.2f}%")

print(f"\nTraining complete.")
print(f"Best Test Accuracy: {best_acc:.2f}% at epoch {best_epoch}")

Epoch 01/50 | Train Loss: 3.5791 Acc: 11.42% | Test Loss: 2.7025 Acc: 17.86%
  → New best model saved! Test Acc: 17.86%
Epoch 02/50 | Train Loss: 2.5516 Acc: 21.09% | Test Loss: 2.4749 Acc: 19.56%
  → New best model saved! Test Acc: 19.56%
Epoch 03/50 | Train Loss: 2.2422 Acc: 27.21% | Test Loss: 2.2125 Acc: 26.19%
  → New best model saved! Test Acc: 26.19%
Epoch 04/50 | Train Loss: 2.0513 Acc: 30.42% | Test Loss: 2.0467 Acc: 29.93%
  → New best model saved! Test Acc: 29.93%
Epoch 05/50 | Train Loss: 1.8552 Acc: 34.01% | Test Loss: 1.9887 Acc: 31.29%
  → Checkpoint saved: epoch_05.pt
  → New best model saved! Test Acc: 31.29%
Epoch 06/50 | Train Loss: 1.7366 Acc: 38.68% | Test Loss: 1.9099 Acc: 31.63%
  → New best model saved! Test Acc: 31.63%
Epoch 07/50 | Train Loss: 1.6113 Acc: 41.45% | Test Loss: 1.8596 Acc: 31.97%
  → New best model saved! Test Acc: 31.97%
Epoch 08/50 | Train Loss: 1.5166 Acc: 43.54% | Test Loss: 1.6524 Acc: 36.73%
  → New best model saved! Test Acc: 36.73%
Epoch 

In [10]:
# Final verification and per-class accuracy

# UPDATED PATH to load from new directory
best_path = os.path.join(CHECKPOINT_DIR, 'best_stgcn_hri30.pt')
checkpoint = torch.load(best_path, map_location=device, weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

all_preds  = []
all_labels = []

with torch.no_grad():
    for batch_x, batch_y in test_loader:
        batch_x = batch_x.to(device)
        output = model(batch_x)
        _, predicted = output.max(1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(batch_y.numpy())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)

overall_acc = 100.0 * (all_preds == all_labels).mean()
print(f"Overall Test Accuracy (best model): {overall_acc:.2f}%")

print("\nPer-class accuracy:")
for c in range(30):
    mask = all_labels == c
    class_acc = 100.0 * (all_preds[mask] == all_labels[mask]).mean()
    print(f"  Class {c:02d}: {class_acc:.1f}%")

# Save results to Drive
results_dir = '/content/drive/MyDrive/HRC_Research/results/accuracy_logs'
os.makedirs(results_dir, exist_ok=True)
# UPDATED FILENAME
results_path = os.path.join(results_dir, 'stgcn_hri30_70_10_20_results.txt')

with open(results_path, 'w') as f:
    f.write(f"ST-GCN Fine-tuned on HRI30 (70/10/20 Split)\n")
    f.write(f"Best epoch: {checkpoint['epoch']}\n")
    f.write(f"Overall Test Accuracy: {overall_acc:.2f}%\n\n")
    f.write("Per-class accuracy:\n")
    for c in range(30):
        mask = all_labels == c
        class_acc = 100.0 * (all_preds[mask] == all_labels[mask]).mean()
        f.write(f"  Class {c:02d}: {class_acc:.1f}%\n")

print(f"\nResults saved to Drive: stgcn_hri30_70_10_20_results.txt")
print("\n=== PHASE 2.3 COMPLETE ===")

Overall Test Accuracy (best model): 55.27%

Per-class accuracy:
  Class 00: 31.6%
  Class 01: 40.0%
  Class 02: 20.0%
  Class 03: 70.0%
  Class 04: 63.2%
  Class 05: 45.0%
  Class 06: 75.0%
  Class 07: 40.0%
  Class 08: 42.1%
  Class 09: 31.6%
  Class 10: 60.0%
  Class 11: 55.0%
  Class 12: 57.9%
  Class 13: 75.0%
  Class 14: 25.0%
  Class 15: 47.4%
  Class 16: 63.2%
  Class 17: 94.7%
  Class 18: 50.0%
  Class 19: 55.0%
  Class 20: 80.0%
  Class 21: 45.0%
  Class 22: 60.0%
  Class 23: 94.7%
  Class 24: 78.9%
  Class 25: 25.0%
  Class 26: 95.0%
  Class 27: 100.0%
  Class 28: 21.1%
  Class 29: 20.0%

Results saved to Drive: stgcn_hri30_70_10_20_results.txt

=== PHASE 2.3 COMPLETE ===
